In [ ]:
!pip install keras-tuner # Install the keras-tuner package
import tensorflow as tf
import keras_tuner as kt # Now you can import keras_tuner
from tensorflow.keras import layers, models
from sklearn.metrics import classification_report
import numpy as np

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.1/129.1 kB 10.4 MB/s eta 0:00:00


In [ ]:
# CNN Feature Extractor
def build_cnn_model(input_shape):
    inputs = layers.Input(shape=input_shape)
    x = layers.Conv2D(32, (3, 3), activation="relu", padding="same")(inputs)
    x = layers.MaxPooling2D((2, 2))(x)
    x = layers.Conv2D(64, (3, 3), activation="relu", padding="same")(x)
    x = layers.MaxPooling2D((2, 2))(x)
    x = layers.Conv2D(128, (3, 3), activation="relu", padding="same")(x)
    x = layers.MaxPooling2D((2, 2))(x)
    x = layers.Flatten()(x)

    cnn_model = models.Model(inputs, x, name="CNN_Model")
    return cnn_model

In [ ]:
# Transformer Block
def build_transformer_block(sequence_length, embed_dim=128, num_heads=4, ff_dim=128):
    inputs = layers.Input(shape=(sequence_length, embed_dim))
    x = layers.MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim)(inputs, inputs)
    x = layers.LayerNormalization(epsilon=1e-6)(x)
    x = layers.Dense(ff_dim, activation="relu")(x)
    x = layers.Dense(embed_dim)(x)
    x = layers.LayerNormalization(epsilon=1e-6)(x)
    x = layers.GlobalAveragePooling1D()(x)

    transformer_model = models.Model(inputs, x, name="Transformer_Block")
    return transformer_model

In [ ]:
# Fusion Model (CNN + Transformer)
def build_cnn_transformer_fusion(input_shape, sequence_length, num_classes):
    cnn_model = build_cnn_model(input_shape)
    transformer_model = build_transformer_block(sequence_length)

    # Merge CNN and Transformer outputs
    merged = layers.Concatenate()([cnn_model.output, transformer_model.output])
    x = layers.Dense(128, activation="relu")(merged)
    output = layers.Dense(num_classes, activation="softmax")(x)

    model = models.Model(inputs=[cnn_model.input, transformer_model.input], outputs=output, name="CNN_Transformer_Fusion")
    return model

In [ ]:
# Example Usage
input_shape = (128, 128, 3)
sequence_length = 10
num_classes = 2

fusion_model = build_cnn_transformer_fusion(input_shape, sequence_length, num_classes)
fusion_model.summary()

Model: "CNN_Transformer_Fusion"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 128, 128,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d (Conv2D)     │ (None, 128, 128,  │        896 │ input_layer[0][0] │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_1       │ (None, 10, 128)   │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d       │ (None, 64, 64,    │          0 │ conv2d[0][0]      │
│ (MaxPooling2D)      │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 10, 128)   │    263,808 │ input_layer_1[0]… │
│ (MultiHeadAttentio… │                   │            │ input_layer_1[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_1 (Conv2D)   │ (None, 64, 64,    │     18,496 │ max_pooling2d[0]… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalization │ (None, 10, 128)   │        256 │ multi_head_atten… │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_1     │ (None, 32, 32,    │          0 │ conv2d_1[0][0]    │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 10, 128)   │     16,512 │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_2 (Conv2D)   │ (None, 32, 32,    │     73,856 │ max_pooling2d_1[… │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 10, 128)   │     16,512 │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_2     │ (None, 16, 16,    │          0 │ conv2d_2[0][0]    │
│ (MaxPooling2D)      │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 10, 128)   │        256 │ dense_1[0][0]     │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten (Flatten)   │ (None, 32768)     │          0 │ max_pooling2d_2[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 128)       │          0 │ layer_normalizat… │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 32896)     │          0 │ flatten[0][0],    │
│ (Concatenate)       │                   │            │ global_average_p… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 128)       │  4,210,816 │ concatenate[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_3 (Dense)     │ (None, 2)         │        258 │ dense_2[0][0]     │
└─────────────────────┴───────────────────┴────────────┴─────────────────

 Total params: 4,601,666 (17.55 MB)

 Trainable params: 4,601,666 (17.55 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
import tensorflow as tf
import keras_tuner as kt
from tensorflow.keras import layers, models
from sklearn.metrics import classification_report
import numpy as np
import shutil

shutil.rmtree("tuner_results", ignore_errors=True)

# CNN Model
def build_cnn_model(hp, input_shape):
    inputs = layers.Input(shape=input_shape)

    filters1 = hp.Choice("filters1", [32, 64, 128])
    filters2 = hp.Choice("filters2", [64, 128, 256])
    filters3 = hp.Choice("filters3", [128, 256, 512])

    x = layers.Conv2D(filters1, (3, 3), activation="relu", padding="same")(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D((2, 2))(x)

    x = layers.Conv2D(filters2, (3, 3), activation="relu", padding="same")(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D((2, 2))(x)

    x = layers.Conv2D(filters3, (3, 3), activation="relu", padding="same")(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D((2, 2))(x)

    x = layers.Flatten()(x)
    return models.Model(inputs, x, name="CNN_Model")

# Transformer Block
def build_transformer_block(hp, embed_dim, sequence_length):
    num_heads = hp.Choice("num_heads", [2, 4])
    ff_dim = hp.Choice("ff_dim", [64, 128])

    inputs = layers.Input(shape=(sequence_length, embed_dim))
    attention = layers.MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim)(inputs, inputs)
    x = layers.Add()([inputs, attention])
    x = layers.LayerNormalization(epsilon=1e-6)(x)

    ff = layers.Dense(ff_dim, activation="relu")(x)
    ff = layers.Dense(embed_dim)(ff)
    x = layers.Add()([x, ff])
    x = layers.LayerNormalization(epsilon=1e-6)(x)

    x = layers.GlobalAveragePooling1D()(x)
    return models.Model(inputs, x, name="Transformer_Block")

# Fusion Model
def build_cnn_transformer_fusion(hp, input_shape, sequence_length, num_classes):
    cnn_model = build_cnn_model(hp, input_shape)
    transformer_model = build_transformer_block(hp, embed_dim=128, sequence_length=sequence_length)

    merged = layers.Concatenate()([cnn_model.output, transformer_model.output])

    dense_units = hp.Choice("dense_units", [64, 128])
    dropout_rate = hp.Float("dropout_rate", 0.3, 0.6, step=0.1)

    x = layers.Dense(dense_units, activation="relu")(merged)
    x = layers.Dropout(dropout_rate)(x)

    output = layers.Dense(num_classes, activation="softmax")(x)

    model = models.Model(inputs=[cnn_model.input, transformer_model.input], outputs=output)
    model.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])
    return model

# Structured synthetic data generator
def generate_structured_data(num_samples, input_shape, sequence_length, num_classes):
    labels = np.array([0] * (num_samples // 2) + [1] * (num_samples // 2))
    np.random.shuffle(labels)

    # Class 0: low mean; Class 1: high mean
    X_images = np.random.normal(loc=labels[:, None, None, None] * 2.0, scale=1.0, size=(num_samples, *input_shape)).astype(np.float32)
    X_seq = np.random.normal(loc=labels[:, None, None] * 2.0, scale=1.0, size=(num_samples, sequence_length, 128)).astype(np.float32)
    y = tf.keras.utils.to_categorical(labels, num_classes)
    return X_images, X_seq, y, labels

# Training pipeline
def run_hyperparameter_tuning(input_shape, sequence_length, num_classes):
    tuner = kt.RandomSearch(
        lambda hp: build_cnn_transformer_fusion(hp, input_shape, sequence_length, num_classes),
        objective="val_accuracy",
        max_trials=5,
        directory="tuner_results",
        project_name="cnn_transformer_tuning"
    )

    X_images, X_seq, y, _ = generate_structured_data(300, input_shape, sequence_length, num_classes)

    early_stop = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

    tuner.search([X_images, X_seq], y, validation_split=0.2, epochs=10, callbacks=[early_stop])

    best_model = tuner.get_best_models(1)[0]

    # Test data with same structured pattern
    X_test_image, X_test_seq, y_test, y_true = generate_structured_data(40, input_shape, sequence_length, num_classes)
    y_pred = best_model.predict([X_test_image, X_test_seq])
    return y_pred, y_test, y_true

# Run it!
y_pred, y_test, y_true = run_hyperparameter_tuning((128, 128, 3), 10, 2)

y_pred_classes = np.argmax(y_pred, axis=1)
print(classification_report(y_true, y_pred_classes, digits=4, zero_division=1))


Trial 5 Complete [00h 00m 23s]
val_accuracy: 1.0

Best val_accuracy So Far: 1.0
Total elapsed time: 00h 02m 20s


/usr/local/lib/python3.12/dist-packages/keras/src/saving/saving_lib.py:802: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 66 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


2/2 ━━━━━━━━━━━━━━━━━━━━ 3s 2s/step
              precision    recall  f1-score   support

           0     1.0000    1.0000    1.0000        20
           1     1.0000    1.0000    1.0000        20

    accuracy                         1.0000        40
   macro avg     1.0000    1.0000    1.0000        40
weighted avg     1.0000    1.0000    1.0000        40



In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
from sklearn.utils import class_weight

# Set seed for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

# -------------------------------
# CNN Feature Extractor with Regularization
# -------------------------------
def build_cnn_model(input_shape):
    inputs = layers.Input(shape=input_shape)
    x = layers.Conv2D(32, (3, 3), activation="relu", padding="same",
                      kernel_regularizer=keras.regularizers.l2(0.001))(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D((2, 2))(x)

    x = layers.Conv2D(64, (3, 3), activation="relu", padding="same",
                      kernel_regularizer=keras.regularizers.l2(0.001))(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D((2, 2))(x)

    x = layers.Conv2D(128, (3, 3), activation="relu", padding="same",
                      kernel_regularizer=keras.regularizers.l2(0.001))(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D((2, 2))(x)

    x = layers.Flatten()(x)
    return models.Model(inputs, x, name="CNN_Model")

# -------------------------------
# Transformer Block
# -------------------------------
def build_transformer_block(sequence_length, embed_dim=128, num_heads=4, ff_dim=128):
    inputs = layers.Input(shape=(sequence_length, embed_dim))

    attn_output = layers.MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim)(inputs, inputs)
    attn_output = layers.Dropout(0.1)(attn_output)
    out1 = layers.LayerNormalization(epsilon=1e-6)(inputs + attn_output)

    ffn_output = layers.Dense(ff_dim, activation="relu")(out1)
    ffn_output = layers.Dense(embed_dim)(ffn_output)
    ffn_output = layers.Dropout(0.1)(ffn_output)
    out2 = layers.LayerNormalization(epsilon=1e-6)(out1 + ffn_output)

    x = layers.GlobalAveragePooling1D()(out2)
    return models.Model(inputs, x, name="Transformer_Block")

# -------------------------------
# Fusion Model: CNN + Transformer
# -------------------------------
def build_cnn_transformer_fusion(input_shape, sequence_length, num_classes):
    cnn_model = build_cnn_model(input_shape)
    transformer_model = build_transformer_block(sequence_length)

    merged = layers.Concatenate()([cnn_model.output, transformer_model.output])
    x = layers.Dense(256, activation="relu", kernel_regularizer=keras.regularizers.l2(0.001))(merged)
    x = layers.Dropout(0.5)(x)
    output = layers.Dense(num_classes, activation="softmax")(x)

    model = models.Model(inputs=[cnn_model.input, transformer_model.input], outputs=output, name="CNN_Transformer_Fusion")
    return model

# -------------------------------
# Data Preparation (Structured Synthetic)
# -------------------------------
input_shape = (128, 128, 3)
sequence_length = 10
num_classes = 2

X_img = np.random.rand(2000, 128, 128, 3).astype(np.float32)
X_seq = np.random.rand(2000, 10, 128).astype(np.float32)
y = np.random.randint(0, 2, 2000)

for i in range(2000):
    if y[i] == 0:
        X_img[i, :64, :64] *= 0.3
        X_seq[i] *= 0.3
    else:
        X_img[i, 64:, 64:] += 0.7
        X_seq[i] += 0.7

print("Class distribution:", np.bincount(y))

X_img_train, X_img_test = X_img[:1600], X_img[1600:]
X_seq_train, X_seq_test = X_seq[:1600], X_seq[1600:]
y_train, y_test = y[:1600], y[1600:]

class_weights = class_weight.compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
class_weights_dict = dict(enumerate(class_weights))

# -------------------------------
# Compile & Train
# -------------------------------
model = build_cnn_transformer_fusion(input_shape, sequence_length, num_classes)
model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])

model.summary()

early_stop = tf.keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=6, restore_best_weights=True)

model.fit([X_img_train, X_seq_train], y_train,
          validation_split=0.2,
          epochs=50,
          batch_size=32,
          callbacks=[early_stop],
          class_weight=class_weights_dict,
          verbose=2)

# -------------------------------
# Evaluation with Metrics
# -------------------------------
y_pred_probs = model.predict([X_img_test, X_seq_test])
y_pred = np.argmax(y_pred_probs, axis=1)

acc = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average='binary', zero_division=0)
recall = recall_score(y_test, y_pred, average='binary', zero_division=0)
f1 = f1_score(y_test, y_pred, average='binary', zero_division=0)

print(f"\n✅ Accuracy:  {acc:.4f}")
print(f"📌 Precision: {precision:.4f}")
print(f"📌 Recall:    {recall:.4f}")
print(f"📌 F1 Score:  {f1:.4f}")
print("\n📊 Classification Report:")
print(classification_report(y_test, y_pred, zero_division=0))

Class distribution: [ 974 1026]


Model: "CNN_Transformer_Fusion"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_2       │ (None, 128, 128,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_3       │ (None, 10, 128)   │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_3 (Conv2D)   │ (None, 128, 128,  │        896 │ input_layer_2[0]… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 10, 128)   │    263,808 │ input_layer_3[0]… │
│ (MultiHeadAttentio… │                   │            │ input_layer_3[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 128, 128,  │        128 │ conv2d_3[0][0]    │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_3 (Dropout) │ (None, 10, 128)   │          0 │ multi_head_atten… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_3     │ (None, 64, 64,    │          0 │ batch_normalizat… │
│ (MaxPooling2D)      │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_2 (Add)         │ (None, 10, 128)   │          0 │ input_layer_3[0]… │
│                     │                   │            │ dropout_3[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_4 (Conv2D)   │ (None, 64, 64,    │     18,496 │ max_pooling2d_3[… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 10, 128)   │        256 │ add_2[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 64, 64,    │        256 │ conv2d_4[0][0]    │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_4 (Dense)     │ (None, 10, 128)   │     16,512 │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_4     │ (None, 32, 32,    │          0 │ batch_normalizat… │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_5 (Dense)     │ (None, 10, 128)   │     16,512 │ dense_4[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_5 (Conv2D)   │ (None, 32, 32,    │     73,856 │ max_pooling2d_4[… │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_4 (Dropout) │ (None, 10, 128)   │          0 │ dense_5[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 32, 32,    │        512 │ conv2d_5[0][0]    │
│ (BatchNormalizatio… │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_3 (Add)         │ (None, 10, 128)   │          0 │ layer_normalizat… │
│                     │                   │            │ dropout_4[0][0] 

 Total params: 8,813,634 (33.62 MB)

 Trainable params: 8,813,186 (33.62 MB)

 Non-trainable params: 448 (1.75 KB)

Epoch 1/50
40/40 - 13s - 330ms/step - accuracy: 0.9883 - loss: 0.7955 - val_accuracy: 0.5594 - val_loss: 4.7196
Epoch 2/50
40/40 - 1s - 29ms/step - accuracy: 1.0000 - loss: 0.6904 - val_accuracy: 1.0000 - val_loss: 0.6350
Epoch 3/50
40/40 - 1s - 28ms/step - accuracy: 1.0000 - loss: 0.5903 - val_accuracy: 1.0000 - val_loss: 0.5471
Epoch 4/50
40/40 - 1s - 28ms/step - accuracy: 1.0000 - loss: 0.5117 - val_accuracy: 1.0000 - val_loss: 0.5545
Epoch 5/50
40/40 - 1s - 28ms/step - accuracy: 1.0000 - loss: 0.4482 - val_accuracy: 1.0000 - val_loss: 0.4197
Epoch 6/50
40/40 - 1s - 28ms/step - accuracy: 1.0000 - loss: 0.3952 - val_accuracy: 1.0000 - val_loss: 0.3710
Epoch 7/50
40/40 - 1s - 29ms/step - accuracy: 1.0000 - loss: 0.3503 - val_accuracy: 1.0000 - val_loss: 0.3296
Epoch 8/50
40/40 - 1s - 30ms/step - accuracy: 1.0000 - loss: 0.3118 - val_accuracy: 1.0000 - val_loss: 0.2939
13/13 ━━━━━━━━━━━━━━━━━━━━ 3s 121ms/step

✅ Accuracy:  1.0000
📌 Precision: 1.0000
📌 Recall:    1.0000
📌 F1 Score:  1.0

In [ ]:
!pip install -q keras-tuner

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
from sklearn.metrics import classification_report
import numpy as np

# ------------------------------
# CNN Feature Extractor
# ------------------------------
def build_cnn_model(input_shape):
    inputs = layers.Input(shape=input_shape)
    x = layers.Conv2D(32, (3, 3), activation="relu", padding="same")(inputs)
    x = layers.MaxPooling2D((2, 2))(x)
    x = layers.Conv2D(64, (3, 3), activation="relu", padding="same")(x)
    x = layers.MaxPooling2D((2, 2))(x)
    x = layers.Conv2D(128, (3, 3), activation="relu", padding="same")(x)
    x = layers.MaxPooling2D((2, 2))(x)
    x = layers.Flatten()(x)
    x = layers.Dropout(0.3)(x)  # Dropout added
    cnn_model = models.Model(inputs, x, name="CNN_Model")
    return cnn_model

# ------------------------------
# Transformer Block
# ------------------------------
def build_transformer_block(sequence_length, embed_dim=64, num_heads=4, ff_dim=64):
    inputs = layers.Input(shape=(sequence_length, embed_dim))
    x = layers.MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim)(inputs, inputs)
    x = layers.Dropout(0.2)(x)  # Dropout added
    x = layers.LayerNormalization(epsilon=1e-6)(x)
    x = layers.Dense(ff_dim, activation="relu")(x)
    x = layers.Dropout(0.2)(x)  # Dropout added
    x = layers.Dense(embed_dim)(x)
    x = layers.LayerNormalization(epsilon=1e-6)(x)
    x = layers.GlobalAveragePooling1D()(x)
    transformer_model = models.Model(inputs, x, name="Transformer_Block")
    return transformer_model

# ------------------------------
# CNN + Transformer Fusion Model
# ------------------------------
def build_cnn_transformer_fusion(input_shape, sequence_length, num_classes):
    cnn_model = build_cnn_model(input_shape)
    transformer_model = build_transformer_block(sequence_length)

    # Merge CNN and Transformer outputs
    merged = layers.Concatenate()([cnn_model.output, transformer_model.output])
    x = layers.Dense(64, activation="relu")(merged)
    x = layers.Dropout(0.3)(x)
    output = layers.Dense(num_classes, activation="softmax")(x)

    model = models.Model(inputs=[cnn_model.input, transformer_model.input], outputs=output, name="CNN_Transformer_Fusion")
    return model

# ------------------------------
# Example Usage
# ------------------------------
input_shape = (128, 128, 3)
sequence_length = 10
num_classes = 2

model = build_cnn_transformer_fusion(input_shape, sequence_length, num_classes)
model.summary()


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.1/129.1 kB 5.4 MB/s eta 0:00:00


Model: "CNN_Transformer_Fusion"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 128, 128,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_1       │ (None, 10, 64)    │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d (Conv2D)     │ (None, 128, 128,  │        896 │ input_layer[0][0] │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 10, 64)    │     66,368 │ input_layer_1[0]… │
│ (MultiHeadAttentio… │                   │            │ input_layer_1[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d       │ (None, 64, 64,    │          0 │ conv2d[0][0]      │
│ (MaxPooling2D)      │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_2 (Dropout) │ (None, 10, 64)    │          0 │ multi_head_atten… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_1 (Conv2D)   │ (None, 64, 64,    │     18,496 │ max_pooling2d[0]… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalization │ (None, 10, 64)    │        128 │ dropout_2[0][0]   │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_1     │ (None, 32, 32,    │          0 │ conv2d_1[0][0]    │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 10, 64)    │      4,160 │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_2 (Conv2D)   │ (None, 32, 32,    │     73,856 │ max_pooling2d_1[… │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_3 (Dropout) │ (None, 10, 64)    │          0 │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_2     │ (None, 16, 16,    │          0 │ conv2d_2[0][0]    │
│ (MaxPooling2D)      │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 10, 64)    │      4,160 │ dropout_3[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten (Flatten)   │ (None, 32768)     │          0 │ max_pooling2d_2[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 10, 64)    │        128 │ dense_1[0][0]     │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 32768)     │          0 │ flatten[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 64)        │          0 │ layer_normalizat… │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 32832)     │          0 │ dropout[0][0],  

 Total params: 2,269,634 (8.66 MB)

 Trainable params: 2,269,634 (8.66 MB)

 Non-trainable params: 0 (0.00 B)